In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, sys

OUTPUTS_SAVE_PATH = os.path.dirname('/content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data/subjects')
sys.path.append(OUTPUTS_SAVE_PATH)
EXCLUDED = ['QPF42', 'USQ95']
SUBJECTS = ['ACB71', 'DGR11', 'HMK96', 'JPY86', 'LRK01', 'LYY64', 'MNY88', 'MRB47', 'NXB64', 'OLW10', 'QFT39', 'QPI83', 'RRO98', 'SAB93', 'SIT48', 'TRA37', 'UJM92', 'WAL74', 'WHR08', 'WJX11']
RANDOM_SEED = 42


In [4]:
import mne
import numpy as np
import pandas as pd
import gc
import random

random.seed(RANDOM_SEED)
print(f"Setup complete with {len(SUBJECTS)} subjects found: {SUBJECTS}")
print(f"OUTPUTS_SAVE_PATH: {OUTPUTS_SAVE_PATH}")


Setup complete with 20 subjects found: ['ACB71', 'DGR11', 'HMK96', 'JPY86', 'LRK01', 'LYY64', 'MNY88', 'MRB47', 'NXB64', 'OLW10', 'QFT39', 'QPI83', 'RRO98', 'SAB93', 'SIT48', 'TRA37', 'UJM92', 'WAL74', 'WHR08', 'WJX11']
OUTPUTS_SAVE_PATH: /content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data


### Combine per-subject files:

In [20]:
all_X, all_y, all_subjects = [], [], []

for subj in SUBJECTS:
    X_s = np.load(f'{OUTPUTS_SAVE_PATH}/subjects/{subj}_X.npy')
    y_s = np.load(f'{OUTPUTS_SAVE_PATH}/subjects/{subj}_y.npy')
    all_X.append(X_s)
    all_y.append(y_s)
    all_subjects.extend([subj] * len(y_s))
    del X_s, y_s

X_full = np.concatenate(all_X)
y_full = np.concatenate(all_y)
subjects_full = np.array(all_subjects)

del all_X, all_y
gc.collect()

print(f"X_full: {X_full.shape}")
print(f"y_full: {y_full.shape}")
print(f"subjects_full: {subjects_full.shape}")


X_full: (31515, 32, 801)
y_full: (31515,)
subjects_full: (31515,)


In [21]:
print(f"Shape: {X_full.shape}")
print(f"Value range: min={X_full.min()}, max={X_full.max()}, mean={X_full.mean()}")
print(f"Class balance: low={(y_full==0).sum()}, high={(y_full==1).sum()}")
print(f"\nPer-subject epoch counts:")
for s in np.unique(subjects_full): 
    mask = subjects_full == s
    print(f"  {s}: {mask.sum()} " 
          f"(low={(y_full[mask]==0).sum()}, high={(y_full[mask]==1).sum()})")

Shape: (31515, 32, 801)
Value range: min=-0.00012170678382972255, max=0.0001267847983399406, mean=4.6116923880219646e-17
Class balance: low=15964, high=15551

Per-subject epoch counts:
  ACB71: 1560 (low=781, high=779)
  DGR11: 1524 (low=790, high=734)
  HMK96: 1688 (low=852, high=836)
  JPY86: 1565 (low=786, high=779)
  LRK01: 1450 (low=726, high=724)
  LYY64: 1687 (low=853, high=834)
  MNY88: 1588 (low=794, high=794)
  MRB47: 1346 (low=699, high=647)
  NXB64: 1694 (low=870, high=824)
  OLW10: 1631 (low=830, high=801)
  QFT39: 1623 (low=807, high=816)
  QPI83: 1611 (low=839, high=772)
  RRO98: 1740 (low=885, high=855)
  SAB93: 1706 (low=856, high=850)
  SIT48: 1215 (low=588, high=627)
  TRA37: 1415 (low=722, high=693)
  UJM92: 1588 (low=814, high=774)
  WAL74: 1690 (low=852, high=838)
  WHR08: 1555 (low=795, high=760)
  WJX11: 1639 (low=825, high=814)


### Slice four time windows:

In [22]:
# Where index i corresponds to time i ms
X_0800 = X_full # full window
X_0200 = X_full[:,:,0:201] # early window
X_300500 = X_full[:,:,300:501] # N400 window 
X_500800 = X_full[:,:,500:801] # late window 

print("Time windows sliced:")
print(f" 0-800 ms (full): {X_0800.shape}")
print(f" 0-200 ms (early): {X_0200.shape}")
print(f" 300-500 ms (N400): {X_300500.shape}")
print(f" 500-800 ms (late): {X_500800.shape}")


Time windows sliced:
 0-800 ms (full): (31515, 32, 801)
 0-200 ms (early): (31515, 32, 201)
 300-500 ms (N400): (31515, 32, 201)
 500-800 ms (late): (31515, 32, 301)


### Seal test subjects: 

In [ ]:
rng = np.random.RandomState(RANDOM_SEED)
all_subs_unique = np.unique(subjects_full).copy()
rng.shuffle(all_subs_unique)

out_sample_test_subjects = all_subs_unique[:2]
train_val_subjects = all_subs_unique[2:]

print("OUT_SAMPLE_TEST SUBJECTS (not to be loaded until final evaluation):")
print(out_sample_test_subjects)
print(f"\nTRAIN_VAL SUBJECTS ({len(train_val_subjects)}):")
print(train_val_subjects)

out_sample_test_mask = np.isin(subjects_full, out_sample_test_subjects)
train_val_mask = np.isin(subjects_full, train_val_subjects)

np.save(f'{OUTPUTS_SAVE_PATH}/out_sample_test_subjects.npy', out_sample_test_subjects)
np.save(f'{OUTPUTS_SAVE_PATH}/train_val_subjects.npy', train_val_subjects)
print("\nSubject lists saved.")

OUT_SAMPLE_TEST SUBJECTS (not to be loaded until final evaluation):
['ACB71' 'WAL74']

TRAIN_VAL SUBJECTS (18):
['TRA37' 'DGR11' 'NXB64' 'LYY64' 'QPI83' 'JPY86' 'WHR08' 'UJM92' 'SAB93'
 'HMK96' 'OLW10' 'WJX11' 'LRK01' 'RRO98' 'MRB47' 'QFT39' 'SIT48' 'MNY88']

Subject lists saved.


### Save train_val and out_sample_test files:

In [24]:
os.makedirs(f'{OUTPUTS_SAVE_PATH}/train_val', exist_ok=True)
os.makedirs(f'{OUTPUTS_SAVE_PATH}/out_sample_test', exist_ok=True)

WINDOWS = {
    '0800': X_0800, 
    '0200': X_0200, 
    '300500':X_300500, 
    '500800': X_500800
}

for name, X_win in WINDOWS.items(): 
    np.save(f'{OUTPUTS_SAVE_PATH}/train_val/X_{name}.npy', X_win[train_val_mask])
np.save(f'{OUTPUTS_SAVE_PATH}/train_val/y.npy', y_full[train_val_mask])
np.save(f'{OUTPUTS_SAVE_PATH}/train_val/subjects.npy', subjects_full[train_val_mask])
print("train_val files saved.")


for name, X_win in WINDOWS.items(): 
    np.save(f'{OUTPUTS_SAVE_PATH}/out_sample_test/X_{name}.npy', X_win[out_sample_test_mask])
np.save(f'{OUTPUTS_SAVE_PATH}/out_sample_test/y.npy', y_full[out_sample_test_mask])
np.save(f'{OUTPUTS_SAVE_PATH}/out_sample_test/subjects.npy', subjects_full[out_sample_test_mask])
print("out_sample_test files saved.")
print("Sealed out_sample_test files saved.")


train_val files saved.
out_sample_test files saved.
Sealed out_sample_test files saved.


In [5]:
X_tv = np.load(f'{OUTPUTS_SAVE_PATH}/train_val/X_0800.npy')
y_tv = np.load(f'{OUTPUTS_SAVE_PATH}/train_val/y.npy')
s_tv = np.load(f'{OUTPUTS_SAVE_PATH}/train_val/subjects.npy')

X_te = np.load(f'{OUTPUTS_SAVE_PATH}/out_sample_test/X_0800.npy')
y_te = np.load(f'{OUTPUTS_SAVE_PATH}/out_sample_test/y.npy')
s_te = np.load(f'{OUTPUTS_SAVE_PATH}/out_sample_test/subjects.npy')

print(f"train_val: {X_tv.shape} | low={(y_tv==0).sum()}, high={(y_tv==1).sum()}")
print(f"out_sample_test:     {X_te.shape} | low={(y_te==0).sum()}, high={(y_te==1).sum()}")
print(f"\ntrain_val subjects ({len(np.unique(s_tv))}): {np.unique(s_tv)}")
print(f"out_sample_test subjects     ({len(np.unique(s_te))}): {np.unique(s_te)}")

overlap = set(np.unique(s_tv)) & set(np.unique(s_te))
assert len(overlap) == 0, f"CONTAMINATION DETECTED: {overlap}"
print(f"\nSubject overlap: none ✓")

print("\nWindow timepoint counts (train_val):")
for name, expected_T in [('0800',801),('0200',201),('300500',201),('500800',301)]:
    X_c = np.load(f'{OUTPUTS_SAVE_PATH}/train_val/X_{name}.npy')
    ok  = "✓" if X_c.shape[2] == expected_T else "✗ ERROR"
    print(f"  X_{name}: {X_c.shape} {ok}")
    del X_c

print("\nWindow timepoint counts (out_sample_test):")
for name, expected_T in [('0800',801),('0200',201),('300500',201),('500800',301)]:
    X_c = np.load(f'{OUTPUTS_SAVE_PATH}/out_sample_test/X_{name}.npy')
    ok  = "✓" if X_c.shape[2] == expected_T else "✗ ERROR"
    print(f"  X_{name}: {X_c.shape} {ok}")
    del X_c

print("\nAll verifications passed. Preprocessing pipeline complete.")

train_val: (28265, 32, 801) | low=14331, high=13934
out_sample_test:     (3250, 32, 801) | low=1633, high=1617

train_val subjects (18): ['DGR11' 'HMK96' 'JPY86' 'LRK01' 'LYY64' 'MNY88' 'MRB47' 'NXB64' 'OLW10'
 'QFT39' 'QPI83' 'RRO98' 'SAB93' 'SIT48' 'TRA37' 'UJM92' 'WHR08' 'WJX11']
out_sample_test subjects     (2): ['ACB71' 'WAL74']

Subject overlap: none ✓

Window timepoint counts (train_val):
  X_0800: (28265, 32, 801) ✓
  X_0200: (28265, 32, 201) ✓
  X_300500: (28265, 32, 201) ✓
  X_500800: (28265, 32, 301) ✓

Window timepoint counts (out_sample_test):
  X_0800: (3250, 32, 801) ✓
  X_0200: (3250, 32, 201) ✓
  X_300500: (3250, 32, 201) ✓
  X_500800: (3250, 32, 301) ✓

All verifications passed. Preprocessing pipeline complete.


# Downsampling 1000 Hz to 250 Hz

In [3]:
import sys

proj_root = '/content/drive/MyDrive/Colab_Notebooks/DERCo'
if proj_root not in sys.path: 
    sys.path.insert(0, proj_root)

from pathlib import Path
import gc
import numpy as np

BASE_PATH = Path('/content/drive/MyDrive/Colab_Notebooks/DERCo')
DOWNSAMPLED_PATH = BASE_PATH / 'processed_data_250hz'
INPUTS_PATH = BASE_PATH / 'processed_data_1000hz'

EXPECTED_LENGTHS_250HZ = {
    "X_0800.npy": 201,
    "X_0200.npy": 51,
    "X_300500.npy": 51,
    "X_500800.npy": 76,
}


In [ ]:
from src.data_utils import downsample_eeg

def downsample_one_split(split:str): 
    for fname, expected_t in EXPECTED_LENGTHS_250HZ.items(): 
        X = np.load(INPUTS_PATH / split / fname)
        X_250 = downsample_eeg(X)
        
        if X_250.shape[-1] != expected_t:
            raise ValueError(
                f"{fname}: expected {expected_t} timepoints after downsampling, "
                f"got {X_250.shape[-1]}. Original shape was {X.shape}."
            )

        np.save(DOWNSAMPLED_PATH / split / fname, X_250)
        print(f"Saved {DOWNSAMPLED_PATH}/ {split} / {fname}: {X.shape} -> {X_250.shape}")

        del X_250
        gc.collect()

downsample_one_split('out_sample_test')
downsample_one_split('train_val')

Saved /content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data_250hz/ out_sample_test / X_0800.npy: (3250, 32, 801) -> (3250, 32, 201)
Saved /content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data_250hz/ out_sample_test / X_0200.npy: (3250, 32, 201) -> (3250, 32, 51)
Saved /content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data_250hz/ out_sample_test / X_300500.npy: (3250, 32, 201) -> (3250, 32, 51)
Saved /content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data_250hz/ out_sample_test / X_500800.npy: (3250, 32, 301) -> (3250, 32, 76)
Saved /content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data_250hz/ train_val / X_0800.npy: (28265, 32, 801) -> (28265, 32, 201)
Saved /content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data_250hz/ train_val / X_0200.npy: (28265, 32, 201) -> (28265, 32, 51)
Saved /content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data_250hz/ train_val / X_300500.npy: (28265, 32, 201) -> (28265, 32, 51)
Saved /content/drive/MyDrive/Colab_Notebooks/

In [9]:
X_tv = np.load(DOWNSAMPLED_PATH / 'train_val/X_0800.npy')
y_tv = np.load(DOWNSAMPLED_PATH / 'train_val/y.npy')
s_tv = np.load(DOWNSAMPLED_PATH / 'train_val/subjects.npy')

X_te = np.load(DOWNSAMPLED_PATH / 'out_sample_test/X_0800.npy')
y_te = np.load(DOWNSAMPLED_PATH / 'out_sample_test/y.npy')
s_te = np.load(DOWNSAMPLED_PATH / 'out_sample_test/subjects.npy')

print(f"train_val: {X_tv.shape} | low={(y_tv==0).sum()}, high={(y_tv==1).sum()}")
print(f"out_sample_test:     {X_te.shape} | low={(y_te==0).sum()}, high={(y_te==1).sum()}")
print(f"\ntrain_val subjects ({len(np.unique(s_tv))}): {np.unique(s_tv)}")
print(f"out_sample_test subjects     ({len(np.unique(s_te))}): {np.unique(s_te)}")

overlap = set(np.unique(s_tv)) & set(np.unique(s_te))
assert len(overlap) == 0, f"CONTAMINATION DETECTED: {overlap}"
print(f"\nSubject overlap: none ✓")

print("\nWindow timepoint counts (train_val):")
for name, expected_T in EXPECTED_LENGTHS_250HZ.items():
    X_c = np.load(DOWNSAMPLED_PATH / f'train_val/{name}')
    ok  = "✓" if X_c.shape[2] == expected_T else "✗ ERROR"
    print(f"  X_{name}: {X_c.shape} {ok}")
    del X_c

print("\nWindow timepoint counts (out_sample_test):")
for name, expected_T in EXPECTED_LENGTHS_250HZ.items():
    X_c = np.load(DOWNSAMPLED_PATH / f'out_sample_test/{name}')
    ok  = "✓" if X_c.shape[2] == expected_T else "✗ ERROR"
    print(f"  X_{name}: {X_c.shape} {ok}")
    del X_c

print("\nAll verifications passed. Preprocessing pipeline complete.")

train_val: (28265, 32, 201) | low=14331, high=13934
out_sample_test:     (3250, 32, 201) | low=1633, high=1617

train_val subjects (18): ['DGR11' 'HMK96' 'JPY86' 'LRK01' 'LYY64' 'MNY88' 'MRB47' 'NXB64' 'OLW10'
 'QFT39' 'QPI83' 'RRO98' 'SAB93' 'SIT48' 'TRA37' 'UJM92' 'WHR08' 'WJX11']
out_sample_test subjects     (2): ['ACB71' 'WAL74']

Subject overlap: none ✓

Window timepoint counts (train_val):
  X_X_0800.npy: (28265, 32, 201) ✓
  X_X_0200.npy: (28265, 32, 51) ✓
  X_X_300500.npy: (28265, 32, 51) ✓
  X_X_500800.npy: (28265, 32, 76) ✓

Window timepoint counts (out_sample_test):
  X_X_0800.npy: (3250, 32, 201) ✓
  X_X_0200.npy: (3250, 32, 51) ✓
  X_X_300500.npy: (3250, 32, 51) ✓
  X_X_500800.npy: (3250, 32, 76) ✓

All verifications passed. Preprocessing pipeline complete.
